<a href="https://colab.research.google.com/github/merbiji/Computational-Physics/blob/main/PALISBO_PHY161_2_ACTIVITY11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from scipy.linalg import lu
import numpy as np
import itertools

# System equations definition
A_1 = np.array([[2, -6, -1], [-3, -1, 7], [-8, 1, -2]], dtype=float)
A_2 = np.array([[2, 1, -1], [5, 2, 2], [3, 1, 1]], dtype=float)
A_3 = np.array([[2, -4, 5], [1, 1.5, -7], [5, 3, -8]], dtype=float)

k_1 = np.array([38, -34, -20], dtype=float)
k_2 = np.array([1, -4, -5], dtype=float)
k_3 = np.array([36, 7, -31], dtype=float)

systems = [
    (A_1, k_1),
    (A_2, k_2),
    (A_3, k_3)
]

def make_diagonally_dominant(A, k):
    """Reorders the rows of matrix A and vector k to achieve diagonal dominance if possible."""
    n = len(k)
    best_order = None

    # Test all row permutations
    for perm in itertools.permutations(range(n)):
        A_perm = A[list(perm)]
        diag = np.abs(np.diag(A_perm))
        off_diag = np.sum(np.abs(A_perm), axis=1) - diag
        if np.all(diag >= off_diag):
            best_order = list(perm)
            break

    if best_order is not None:
        return A[best_order], k[best_order], True
    return A, k, False

def LU_decomposition(A, k):
    P, L, U = lu(A)
    P_inv = np.linalg.inv(P)
    L_inv = np.linalg.inv(L)
    U_inv = np.linalg.inv(U)
    x = U_inv.dot(L_inv.dot(P_inv.dot(k)))
    print('P:\n', P)
    print('L:\n', L)
    print('U:\n', U)
    print('LU:\n',np.dot(L, U))
    return x

def gauss_seidel(A, k, max_iter=100, epsilon=0.01, divergence_threshold=1e6):
    # Reorder rows to guarantee convergence if possible
    x = np.zeros_like(k, dtype=float)
    A_dom, k_dom, success = make_diagonally_dominant(A, k)
    if success:
        print("Successfully reordered rows for diagonal dominance.")
    else:
        print("Warning: Could not make the matrix strictly diagonally dominant.")
        return x
    n = len(k_dom)
    x = np.zeros(n)
    converged = False

    print('Iteration results:')
    print(' k,    ' + ',    '.join(f'x{i+1}' for i in range(n)))

    for iteration in range(1, max_iter + 1):
        x_old = np.copy(x)
        for i in range(n):
            # Dynamic GS equation: x[i] = (k[i] - sum(A[i, j]*x[j] for j != i)) / A[i, i]
            sum_ax = sum(A_dom[i, j] * x[j] for j in range(n) if j != i)
            x[i] = (k_dom[i] - sum_ax) / A_dom[i, i]

        # Print iteration status
        print(f"{iteration:2d},  " + ",  ".join(f"{val: .4f}" for val in x))

        # Check if the calculations are blowing up (diverging)
        if np.linalg.norm(x) > divergence_threshold:
            print(f"Early termination: Iterations are diverging rapidly (magnitude exceeded {divergence_threshold:.0e}).")
            break

        # Check tolerance using Euclidean distance
        dx = np.linalg.norm(x - x_old)
        if dx < epsilon:
            converged = True
            print('Converged!')
            break

    if not converged:
        print('Failed to converge.')
    return x

for index, (A, k) in enumerate(systems, start=1):
    print(f"\nSYSTEM {index}")
    print("="*80)
    solution_LU = LU_decomposition(A, k)
    print("Solution LU:")
    for i in range(len(solution_LU)):
        print(f"  x{i+1} = {solution_LU[i]:.6f}")
    print("-"*60)
    print("Running Gauss-Seidel Method:")
    solution_GS = gauss_seidel(A, k)
    print("\nSolution GS (Final State):")
    for i in range(len(solution_GS)):
        print(f"  x{i+1} = {solution_GS[i]:.6f}")
    print("-"*60)
    check = np.linalg.solve(A, k)
    print("\nNumPy Benchmark Solution:")
    for i in range(len(check)):
        print(f"  x{i+1} = {check[i]:.6f}")


SYSTEM 1
P:
 [[0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]]
L:
 [[ 1.          0.          0.        ]
 [-0.25        1.          0.        ]
 [ 0.375       0.23913043  1.        ]]
U:
 [[-8.          1.         -2.        ]
 [ 0.         -5.75       -1.5       ]
 [ 0.          0.          8.10869565]]
LU:
 [[-8.  1. -2.]
 [ 2. -6. -1.]
 [-3. -1.  7.]]
Solution LU:
  x1 = 2.981233
  x2 = -4.632708
  x3 = -4.241287
------------------------------------------------------------
Running Gauss-Seidel Method:
Successfully reordered rows for diagonal dominance.
Iteration results:
 k,    x1,    x2,    x3
 1,   2.5000,  -5.5000,  -4.5714
 2,   2.9554,  -4.5863,  -4.2457
 3,   2.9881,  -4.6297,  -4.2379
 4,   2.9808,  -4.6334,  -4.2416
Converged!

Solution GS (Final State):
  x1 = 2.980765
  x2 = -4.633431
  x3 = -4.241591
------------------------------------------------------------

NumPy Benchmark Solution:
  x1 = 2.981233
  x2 = -4.632708
  x3 = -4.241287

SYSTEM 2
P:
 [[0. 0. 1.]
 [1. 0. 0.]
 [0. 1. 0